# 01 - Preprocessing

Adapted from the Urban Taxonomy / HiMoC pipeline for **Winnipeg, Canada**.

- Source uses EUBUCCO cadastre (EPSG:3035) — we use OBPL buildings (EPSG:3978)
- Source uses Overture streets — we use OSM streets
- Canadian advantage: we have HEIGHT + BLDG_POP from OBPL

Steps:
1. Load buildings, set CRS to EPSG:3978, clip to Winnipeg boundary
2. Clean buildings: make_valid, explode, simplify, merge overlaps (per source `generate_buildings.py`)
3. Load roads, filter by approved classes (per source `generate_streets.py`), reproject
4. Compute basic building geometric features
5. Save preprocessed data

In [1]:
import os
os.chdir(os.path.join(os.path.dirname(os.getcwd()), "") if os.path.basename(os.getcwd()) == "notebooks" else ".")
print("Working dir:", os.getcwd())

Working dir: c:\urban-form\urban_taxonomy


In [4]:
import geopandas as gpd
import numpy as np
import pandas as pd
import momepy as mm
import geoplanar
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

OUT = Path("outputs/01_preprocessed")
OUT.mkdir(parents=True, exist_ok=True)

# EPSG:3978 = NAD83 / Canada Atlas Lambert (native CRS of OBPL data)
# Source uses EPSG:3035 (ETRS89-LAEA) for Europe
TARGET_CRS = "EPSG:3978"

c:\urban-form\urban_taxonomy\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Winnipeg boundary

In [5]:
boundary = gpd.read_file("data/winipeg_boundry.gpkg").to_crs(TARGET_CRS)
boundary_union = boundary.union_all()
print("Boundary loaded, CRS:", boundary.crs)

Boundary loaded, CRS: EPSG:3978


## 2. Load and clean buildings

Following the source `generate_buildings.py` pipeline:
1. `make_valid()` 
2. Explode MultiPolygons, keep only Polygons
3. Simplify geometry (0.1 m tolerance)
4. Drop buildings > 200,000 m^2
5. `geoplanar.merge_overlaps()` — merge buildings overlapping >10% or < 25 m^2
6. `geoplanar.trim_overlaps()` — remove remaining overlaps
7. Merge touching collapsing buildings
8. Final simplify + make_valid

In [6]:
buildings = gpd.read_file("data/winnipeg_buildings.gpkg")
buildings = buildings.set_crs(TARGET_CRS, allow_override=True)

# Clip to Winnipeg
buildings = buildings[buildings.intersects(boundary_union)].copy()
buildings = buildings.reset_index(drop=True)
initial_count = len(buildings)
print(f"Loaded {initial_count:,} buildings within Winnipeg")

Loaded 269,289 buildings within Winnipeg


In [7]:
# --- Source pipeline: process_region_buildings() ---
SIMPLIFICATION_TOLERANCE = 0.1  # metres (same as source)
MERGE_LIMIT = 25                # m^2 (same as source)

# 1. Fix invalid geometry
buildings["geometry"] = buildings.make_valid()

# 2. Explode + keep only Polygons
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 3. Simplify
buildings["geometry"] = buildings.simplify(SIMPLIFICATION_TOLERANCE).normalize()

# 4. Drop very large buildings (>200,000 m^2)
buildings = buildings[buildings.area < 200_000].reset_index(drop=True)

# 5. Merge overlapping buildings
buildings = geoplanar.merge_overlaps(buildings, merge_limit=MERGE_LIMIT, overlap_limit=0.1)

# 6. Trim remaining overlaps
buildings = geoplanar.trim_overlaps(buildings, strategy="smallest")

# 7. Explode again + keep Polygons
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 8. Merge touching collapsing buildings (shrink test)
shrink = buildings.buffer(-0.5, resolution=2)
buildings = geoplanar.merge_touching(buildings, np.where(shrink.is_empty), largest=True)
buildings = buildings.explode(ignore_index=True)
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)

# 9. Final simplify + make_valid
buildings["geometry"] = buildings.simplify(SIMPLIFICATION_TOLERANCE)
buildings["geometry"] = buildings.make_valid()
buildings = buildings[buildings.geom_type == "Polygon"].reset_index(drop=True)
buildings["geometry"] = buildings.normalize()

print(f"After cleaning: {len(buildings):,} buildings (dropped {1 - len(buildings)/initial_count:.1%})")

After cleaning: 269,107 buildings (dropped 0.1%)


In [8]:
# Fix nulls in OBPL-specific columns
buildings["HEIGHT"] = buildings["HEIGHT"].fillna(0).clip(lower=0)
buildings["BLDG_POP"] = buildings["BLDG_POP"].fillna(0).clip(lower=0)
print(f"Columns: {list(buildings.columns)}")

Columns: ['BLDG_ID', 'HEIGHT', 'DBUID', 'DB_RATIO', 'DAUID', 'DA_RATIO', 'CSDUID', 'CSD_RATIO', 'CDUID', 'CD_RATIO', 'ORPHAN_RATIO', 'HAS_ORPHAN_BONUS', 'BLDG_POP', 'POP_SOURCE', 'QF_O', 'QF_V', 'id', 'geometry']


## 3. Load and filter roads

Source `generate_streets.py` uses this approved list (service is **excluded**):
```
living_street, motorway, motorway_link, pedestrian, primary, primary_link,
residential, secondary, secondary_link, tertiary, tertiary_link,
trunk, trunk_link, unclassified
```

We follow the same list. Source also runs `neatnet.neatify()` for topology
cleanup but that requires specific Overture schema; we skip it for OSM data.

In [9]:
roads = gpd.read_file("data/winipeg_roads.gpkg").to_crs(TARGET_CRS)

# Source approved_roads list (service excluded)
APPROVED_ROADS = {
    "living_street", "motorway", "motorway_link", "pedestrian",
    "primary", "primary_link", "residential",
    "secondary", "secondary_link",
    "tertiary", "tertiary_link",
    "trunk", "trunk_link", "unclassified",
}
roads = roads[roads["fclass"].isin(APPROVED_ROADS)].copy()
roads = roads[roads.intersects(boundary_union)].copy()
roads = roads.explode(index_parts=False).reset_index(drop=True)

# Sort and clean index
roads = roads.sort_values("osm_id").reset_index(drop=True)
print(f"{len(roads):,} road segments")
print(roads["fclass"].value_counts().to_string())

17,711 road segments
fclass
residential       7679
tertiary          2891
secondary         2661
primary           1468
unclassified      1061
trunk              603
primary_link       423
secondary_link     388
trunk_link         144
motorway           124
tertiary_link      103
pedestrian          91
motorway_link       75


## 4. Save preprocessed data

In [10]:
buildings.to_file(OUT / "buildings.gpkg", driver="GPKG")
roads.to_file(OUT / "roads.gpkg", driver="GPKG")
boundary.to_file(OUT / "boundary.gpkg", driver="GPKG")

print(f"Saved to {OUT}")
print(f"  buildings.gpkg : {len(buildings):,} buildings")
print(f"  roads.gpkg     : {len(roads):,} road segments")
print(f"  boundary.gpkg  : 1 polygon")

Saved to outputs\01_preprocessed
  buildings.gpkg : 269,107 buildings
  roads.gpkg     : 17,711 road segments
  boundary.gpkg  : 1 polygon
